<a href="https://colab.research.google.com/github/mrpunith-1920/gnn-fraud-detection/blob/main/05_gnn_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 9.8 MB/s  0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

import torch, numpy as np, pandas as pd
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, GCNConv, GATConv
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, roc_auc_score

path = '/content/drive/MyDrive/gnn-project/data/elliptic_bitcoin_dataset/'
data = torch.load(path + 'elliptic_graph.pt', weights_only=False)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
data = data.to(device)
print(data, device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], timestep=[203769], train_mask=[203769], test_mask=[203769]) cuda


In [5]:
class GNN(torch.nn.Module):
    def __init__(self, in_dim, hidden=64, kind='sage', layers=2):
        super().__init__()
        Conv = {'gcn': GCNConv, 'sage': SAGEConv, 'gat': GATConv}[kind]
        dims = [in_dim] + [hidden] * layers
        self.convs = torch.nn.ModuleList([Conv(dims[i], dims[i+1]) for i in range(layers)])
        self.out = torch.nn.Linear(hidden, 2)

    def embed(self, x, edge_index):
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
            x = F.dropout(x, 0.3, training=self.training)
        return x

    def forward(self, x, edge_index):
        return self.out(self.embed(x, edge_index))

In [6]:
def train_gnn(kind, epochs=200, hidden=64, lr=0.005, seed=42, layers=2):
    torch.manual_seed(seed)
    model = GNN(data.num_features, hidden, kind, layers).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)

    n_fraud = (data.y[data.train_mask] == 1).sum().item()
    n_legit = (data.y[data.train_mask] == 0).sum().item()
    weights = torch.tensor([1.0, n_legit / n_fraud]).to(device)

    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask], weight=weights)
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        prob = F.softmax(out, dim=1)[:, 1][data.test_mask].cpu().numpy()
    y_true = data.y[data.test_mask].cpu().numpy()
    pred = (prob > 0.5).astype(int)

    row = {
        'Model': kind.upper(), 'Layers': layers, 'Seed': seed,
        'Precision': precision_score(y_true, pred),
        'Recall': recall_score(y_true, pred),
        'F1': f1_score(y_true, pred),
        'PR-AUC': average_precision_score(y_true, prob),
        'ROC-AUC': roc_auc_score(y_true, prob),
    }
    return model, row

In [7]:
model_gat, row_gat = train_gnn('gat')
print(row_gat)

{'Model': 'GAT', 'Layers': 2, 'Seed': 42, 'Precision': 0.7179215270413574, 'Recall': 0.6251154201292706, 'F1': 0.6683119447186574, 'PR-AUC': np.float64(0.5718808931431835), 'ROC-AUC': np.float64(0.9049429819970368)}


In [8]:
rows = []
for kind in ['sage', 'gcn', 'gat']:
    for seed in [0, 1, 2]:
        _, r = train_gnn(kind, seed=seed)
        rows.append(r)
        print(kind, seed, round(r['F1'], 3))

runs = pd.DataFrame(rows)
runs.to_csv(path + 'gnn_runs.csv', index=False)

summary = runs.groupby('Model')[['Precision','Recall','F1','PR-AUC','ROC-AUC']].agg(['mean','std']).round(3)
summary.to_csv(path + 'gnn_summary.csv')
summary

sage 0 0.656
sage 1 0.678
sage 2 0.663
gcn 0 0.526
gcn 1 0.485
gcn 2 0.501
gat 0 0.656
gat 1 0.604
gat 2 0.646


Precision        Recall            F1        PR-AUC        ROC-AUC  \
           mean    std   mean    std   mean    std   mean    std    mean   
Model                                                                      
GAT       0.657  0.065  0.618  0.007  0.636  0.028  0.526  0.025   0.902   
GCN       0.484  0.048  0.528  0.013  0.504  0.020  0.516  0.021   0.839   
SAGE      0.720  0.023  0.619  0.002  0.666  0.011  0.630  0.022   0.907   

              
         std  
Model         
GAT    0.002  
GCN    0.002  
SAGE   0.001

In [9]:
layer_rows = []
for L in [1, 2, 3]:
    _, r = train_gnn('sage', layers=L)
    layer_rows.append(r)

layers_df = pd.DataFrame(layer_rows)[['Layers','Precision','Recall','F1','PR-AUC','ROC-AUC']].round(3)
layers_df.to_csv(path + 'layers_results.csv', index=False)
layers_df

,Layers,Precision,Recall,F1,PR-AUC,ROC-AUC
0,1,0.521,0.636,0.573,0.553,0.900
1,2,0.733,0.625,0.675,0.644,0.907
2,3,0.838,0.608,0.705,0.688,0.901


In [10]:
model_sage, _ = train_gnn('sage', seed=42)
model_sage.eval()
with torch.no_grad():
    emb = model_sage.embed(data.x, data.edge_index).cpu().numpy()

tr = data.train_mask.cpu().numpy()
te = data.test_mask.cpu().numpy()
X_all = data.x.cpu().numpy()
y_all = data.y.cpu().numpy()

X_tr = np.hstack([X_all[tr], emb[tr]]); y_tr = y_all[tr]
X_te = np.hstack([X_all[te], emb[te]]); y_te = y_all[te]

from xgboost import XGBClassifier
ratio = (y_tr == 0).sum() / (y_tr == 1).sum()
hyb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                    scale_pos_weight=ratio, random_state=42, eval_metric='logloss')
hyb.fit(X_tr, y_tr)

pred = hyb.predict(X_te); prob = hyb.predict_proba(X_te)[:, 1]
hybrid_row = {'Model': 'SAGE + XGBoost',
              'Precision': precision_score(y_te, pred), 'Recall': recall_score(y_te, pred),
              'F1': f1_score(y_te, pred), 'PR-AUC': average_precision_score(y_te, prob),
              'ROC-AUC': roc_auc_score(y_te, prob)}
print({k: (round(v, 3) if not isinstance(v, str) else v) for k, v in hybrid_row.items()})

{'Model': 'SAGE + XGBoost', 'Precision': 0.812, 'Recall': 0.513, 'F1': 0.629, 'PR-AUC': np.float64(0.653), 'ROC-AUC': np.float64(0.886)}


In [11]:
torch.save(model_sage.state_dict(), path + 'graphsage.pth')